# 09. refrigerator 보강 이미지 수집 (Naver Shopping API → staging)

**목적**: `config.NAVER_SEARCH_QUERIES['refrigerator']` 쿼리로 보강 이미지 수집

**배경**: v3 재학습 결과 refrigerator per-class accuracy 88.2% (오답 2건 refrigerator→washer_dryer).
단문/소형/냉동고 유형 보강이 목표.

**흐름**: Naver API → `data/staging/refrigerator/{query}/` → (검수) → `data/processed/refrigerator/`

**수집 후 할 일**: `08_approve_staging.ipynb`의 `CLASS_NAME = 'refrigerator'`로 변경 후 검수.

> `data/processed/`와 `data/raw/`는 이 노트북에서 절대 수정하지 않습니다.

In [1]:
import os, sys, re, requests
from io import BytesIO
from pathlib import Path
from PIL import Image as PILImage
from datetime import datetime
import pandas as pd
from tqdm.auto import tqdm

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
import config

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
NAVER_ID     = os.getenv('NAVER_CLIENT_ID', '')
NAVER_SECRET = os.getenv('NAVER_CLIENT_SECRET', '')
if not NAVER_ID or not NAVER_SECRET:
    raise EnvironmentError('NAVER_CLIENT_ID / NAVER_CLIENT_SECRET 환경변수가 없습니다. .env를 확인하세요.')
print('Naver API 자격증명 확인 OK')

CLASS_NAME       = 'refrigerator'
QUERIES          = config.NAVER_SEARCH_QUERIES[CLASS_NAME]
IMAGES_PER_QUERY = 30      # 쿼리당 수집 수 (최대 100)
MIN_IMG_SIZE     = 150     # 픽셀 — 이 미만은 저장 안 함
STAGING_DIR      = os.path.join('data', 'staging', CLASS_NAME)
METADATA_CSV     = os.path.join(config.METADATA_DIR, 'refrigerator_staging_metadata.csv')

print(f'검색어 {len(QUERIES)}개  |  쿼리당 {IMAGES_PER_QUERY}장  |  최대 {len(QUERIES)*IMAGES_PER_QUERY}장')
for i, q in enumerate(QUERIES):
    print(f'  {i+1:2d}. {q}')

Naver API 자격증명 확인 OK
검색어 16개  |  쿼리당 30장  |  최대 480장
   1. 냉장고
   2. 스탠드형 냉장고
   3. 양문형 냉장고
   4. 4도어 냉장고
   5. 삼성 냉장고
   6. LG 냉장고
   7. 단문냉장고
   8. 소형냉장고
   9. 원룸냉장고
  10. 미니냉장고
  11. 냉동고
  12. 스탠드형냉동고
  13. 냉장고 정면
  14. 냉장고 제품사진
  15. 삼성 비스포크 냉장고
  16. LG 오브제 냉장고


In [2]:
def fetch_items(query, display):
    r = requests.get(
        'https://openapi.naver.com/v1/search/shop.json',
        headers={'X-Naver-Client-Id': NAVER_ID, 'X-Naver-Client-Secret': NAVER_SECRET},
        params={'query': query, 'display': display, 'start': 1},
        timeout=config.DOWNLOAD_TIMEOUT,
    )
    r.raise_for_status()
    return r.json().get('items', [])

def download_and_validate(url, save_path, min_px):
    try:
        r = requests.get(url, timeout=config.DOWNLOAD_TIMEOUT)
        if r.status_code != 200 or 'image' not in r.headers.get('Content-Type',''):
            return False, 0, 0
        img = PILImage.open(BytesIO(r.content)).convert('RGB')
        w, h = img.size
        if w < min_px or h < min_px:
            return False, w, h
        img.save(save_path, 'JPEG', quality=95)
        return True, w, h
    except Exception:
        return False, 0, 0

records = []
n_saved = 0
n_skip  = 0
n_exist = 0

os.makedirs(config.METADATA_DIR, exist_ok=True)

for q_i, query in enumerate(QUERIES, 1):
    slug  = re.sub(r'[^\w가-힣]', '_', query)
    qdir  = os.path.join(STAGING_DIR, slug)
    os.makedirs(qdir, exist_ok=True)

    try:
        items = fetch_items(query, IMAGES_PER_QUERY)
    except Exception as e:
        print(f'  [{q_i:2d}/{len(QUERIES)}] {query}: API 오류 -> {e}')
        continue

    saved_q = 0
    exist_q = 0
    skip_q  = 0

    for idx, item in enumerate(items):
        img_url = item.get('image', '')
        if not img_url:
            skip_q += 1
            continue
        fname     = f'stg_{idx:04d}.jpg'
        save_path = os.path.join(qdir, fname)
        title_clean = re.sub(r'<[^>]+>', '', item.get('title', ''))

        if os.path.exists(save_path):
            try:
                img = PILImage.open(save_path)
                w, h = img.size
            except Exception:
                w, h = 0, 0
            records.append({
                'query': query, 'title': title_clean[:120],
                'link': item.get('link',''), 'image_url': img_url,
                'saved_path': save_path, 'width': w, 'height': h,
                'status': 'staged',
                'collected_at': datetime.now().isoformat(timespec='seconds'),
            })
            exist_q += 1
            n_exist += 1
            continue

        ok, w, h = download_and_validate(img_url, save_path, MIN_IMG_SIZE)
        if ok:
            records.append({
                'query': query, 'title': title_clean[:120],
                'link': item.get('link',''), 'image_url': img_url,
                'saved_path': save_path, 'width': w, 'height': h,
                'status': 'staged',
                'collected_at': datetime.now().isoformat(timespec='seconds'),
            })
            saved_q += 1
            n_saved += 1
        else:
            skip_q += 1
            n_skip += 1

    print(f'  [{q_i:2d}/{len(QUERIES)}] {query:20s} -> 신규 {saved_q:2d}장  기존 {exist_q:2d}장  스킵 {skip_q:2d}장')

print(f'\n수집 완료 -- 신규 {n_saved}장  기존 {n_exist}장  스킵(소형/오류) {n_skip}장')

  [ 1/16] 냉장고                  -> 신규 30장  기존  0장  스킵  0장


  [ 2/16] 스탠드형 냉장고             -> 신규 30장  기존  0장  스킵  0장


  [ 3/16] 양문형 냉장고              -> 신규 30장  기존  0장  스킵  0장


  [ 4/16] 4도어 냉장고              -> 신규 30장  기존  0장  스킵  0장


  [ 5/16] 삼성 냉장고               -> 신규 30장  기존  0장  스킵  0장


...

  [12/16] 스탠드형냉동고              -> 신규 30장  기존  0장  스킵  0장


  [13/16] 냉장고 정면               -> 신규 30장  기존  0장  스킵  0장


  [14/16] 냉장고 제품사진             -> 신규 30장  기존  0장  스킵  0장


  [15/16] 삼성 비스포크 냉장고          -> 신규 30장  기존  0장  스킵  0장


  [16/16] LG 오브제 냉장고           -> 신규 30장  기존  0장  스킵  0장

수집 완료 -- 신규 480장  기존 0장  스킵(소형/오류) 0장


In [3]:
new_df = pd.DataFrame(records)
if os.path.exists(METADATA_CSV):
    old_df = pd.read_csv(METADATA_CSV, encoding='utf-8-sig')
    combined = pd.concat([old_df, new_df], ignore_index=True)
    combined.drop_duplicates(subset=['image_url'], keep='last', inplace=True)
else:
    combined = new_df

combined.to_csv(METADATA_CSV, index=False, encoding='utf-8-sig')
print(f'메타데이터 저장: {METADATA_CSV}  ({len(combined)}건)')

print()
print('=== staging 디렉토리 현황 ===')
total_staged = 0
for slug in sorted(os.listdir(STAGING_DIR)):
    d = os.path.join(STAGING_DIR, slug)
    if not os.path.isdir(d):
        continue
    imgs = [f for f in os.listdir(d) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    print(f'  {slug:40s}: {len(imgs):3d}장')
    total_staged += len(imgs)
print(f'  {"합계":40s}: {total_staged:3d}장')

메타데이터 저장: data\metadata\refrigerator_staging_metadata.csv  (480건)

=== staging 디렉토리 현황 ===
  4도어_냉장고                                 :  30장
  LG_냉장고                                  :  30장
  LG_오브제_냉장고                              :  30장
  냉동고                                     :  30장
  냉장고                                     :  30장
  냉장고_정면                                  :  30장
  냉장고_제품사진                                :  30장
  단문냉장고                                   :  30장
  미니냉장고                                   :  30장
  삼성_냉장고                                  :  30장
  삼성_비스포크_냉장고                             :  30장
  소형냉장고                                   :  30장
  스탠드형_냉장고                                :  30장
  스탠드형냉동고                                 :  30장
  양문형_냉장고                                 :  30장
  원룸냉장고                                   :  30장
  합계                                      : 480장


In [4]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

staged_files = []
for q_slug in sorted(os.listdir(STAGING_DIR)):
    qd = os.path.join(STAGING_DIR, q_slug)
    if not os.path.isdir(qd):
        continue
    for f in sorted(os.listdir(qd)):
        if f.lower().endswith(('.jpg','.jpeg','.png')):
            staged_files.append((q_slug, os.path.join(qd, f)))

print(f'스테이징 총 {len(staged_files)}장')

by_query = {}
for slug, fpath in staged_files:
    by_query.setdefault(slug, []).append(fpath)

SAMPLE = 5
n_queries = len(by_query)
fig, axes = plt.subplots(n_queries, SAMPLE, figsize=(SAMPLE*3, n_queries*3))
if n_queries == 1:
    axes = [axes]
axes = np.array(axes)

for row_i, (slug, paths) in enumerate(sorted(by_query.items())):
    for col_i in range(SAMPLE):
        ax = axes[row_i, col_i]
        if col_i < len(paths):
            try:
                ax.imshow(PILImage.open(paths[col_i]).convert('RGB'))
                ax.set_title(f'{slug[:18]}\n{os.path.basename(paths[col_i])}', fontsize=6)
            except:
                ax.text(0.5, 0.5, 'ERR', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')

plt.suptitle(f'수집 스테이징 -- 쿼리별 샘플 {SAMPLE}장', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('다음 단계: 08_approve_staging.ipynb의 CLASS_NAME을 refrigerator로 변경 후 검수')

스테이징 총 480장


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 46020 (\N{HANGUL SYLLABLE DO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 50612 (\N{HANGUL SYLLABLE EO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 45257 (\N{HANGUL SYLLABLE NAENG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 51109 (\N{HANGUL SYLLABLE JANG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 44256 (\N{HANGUL SYLLABLE GO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 50724 (\N{HANGUL SYLLABLE O}) missing from 

C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 51221 (\N{HANGUL SYLLABLE JEONG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 47732 (\N{HANGUL SYLLABLE MYEON}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 54408 (\N{HANGUL SYLLABLE PUM}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 49324 (\N{HANGUL SYLLABLE SA}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 51652 (\N{HANGUL SYLLABLE JIN}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 45800 (\N{HANGUL SYLLABLE DAN}) missing 

C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 48708 (\N{HANGUL SYLLABLE BI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 49828 (\N{HANGUL SYLLABLE SEU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 54252 (\N{HANGUL SYLLABLE PO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 53356 (\N{HANGUL SYLLABLE KEU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 49548 (\N{HANGUL SYLLABLE SO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 54805 (\N{HANGUL SYLLABLE HYEONG}) missing fro

다음 단계: 08_approve_staging.ipynb의 CLASS_NAME을 refrigerator로 변경 후 검수


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 50577 (\N{HANGUL SYLLABLE YANG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 50896 (\N{HANGUL SYLLABLE WEON}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 47352 (\N{HANGUL SYLLABLE RUM}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 49688 (\N{HANGUL SYLLABLE SU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 51665 (\N{HANGUL SYLLABLE JIB}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\SSAFY\AppData\Local\Temp\ipykernel_18904\1818845308.py:40: UserWarning: Glyph 53580 (\N{HANGUL SYLLABLE TE}) missing fro